# Transformer Training on GSM8K

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/transformer-project/output'
print(f'Checkpoints will be saved to: {SAVE_DIR}')

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/KKD2005/transformer-project.git
%cd transformer-project
!pip install -q -r requirements.txt

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Generate tokenizer corpus and train tokenizer
import sys
sys.path.insert(0, '.')
sys.path.insert(0, 'data')

from datasets import load_dataset
from generate_dataset import GSM8KDataset

print('Loading GSM8K dataset...')
raw_dataset = load_dataset('openai/gsm8k', 'main')
print(f'Train examples: {len(raw_dataset["train"])}')

# Generate tokenizer training corpus
tmp_ds = GSM8KDataset(raw_dataset, tokenizer=None)
tmp_ds.create_tokenizer_txt('data/tokenizer_corpus.txt')
print('Tokenizer corpus written to data/tokenizer_corpus.txt')

# Train the tokenizer
from tokenizers import ByteLevelBPETokenizer
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(
    files=['data/tokenizer_corpus.txt'],
    vocab_size=8192,
    min_frequency=2,
    special_tokens=['<pad>', '<eos>', '<bos>', '<problem>', '</problem>',
                    '<reasoning>', '</reasoning>', '<answer>', '</answer>']
)
tokenizer.save('data/gsm8k_tokenizer.json')
print('Tokenizer trained and saved to data/gsm8k_tokenizer.json')

In [ ]:
import os; os.environ['OUTPUT_DIR'] = SAVE_DIR

# Run training
!python training/train.py